# HF-RISK: GCS extraction & MIMIC SHAP comparison (Colab)

Runs the heavy analysis on Colab CPU, not on your laptop.

**What this produces** (and why it matters): your project memory records
"SHAP was run only on the Zhang cohort... Running SHAP on MIMIC remains an open
gap," and dissertation Section 6.3 names GCS extraction from MIMIC as
future-work item #1. This notebook produces both.

**Run the cells top to bottom.** Cell 5 is a fast smoke test - do that first,
confirm it passes, then run the full job in cell 6.

**Before anything goes into the preprint or the website:** read
`results/mimic_gcs_run_log.md` and
`results/tables/mimic_gcs_retrain_results.csv` yourself. They state which
baseline was used, whether the GCS confidence interval crosses zero, and what
the `had_icu_stay` control showed. Review the diffs - don't just accept them.

## Cell 1 - runtime check (optional)

Colab is CPU-only by default, which is fine. Uncomment the GPU lines if you
want to switch to a GPU runtime from the Runtime menu.

In [ ]:
# @title Runtime check
!nvidia-smi 2>/dev/null || echo "CPU runtime (expected; this job is CPU-bound)"
import sys, platform
print("python", sys.version.split()[0], "|", platform.machine())

## Cell 2 - get the code

In [ ]:
# Option A: if this is a git repo on GitHub, clone it:
# !git clone <your-repo-url> hf-risk-heart-failure-ml

# Option B: if you uploaded the folder, Colab puts it in /content.
import os
if not os.path.isdir("/content/hf-risk-heart-failure-ml"):
    os.chdir("/content")
    if not os.path.isdir("/content/hf-risk-heart-failure-ml"):
        print("Upload the hf-risk-heart-failure-ml folder to /content, "
              "or uncomment the git clone line above.")
else:
    os.chdir("/content/hf-risk-heart-failure-ml")

print("cwd:", os.getcwd())
print("scripts present:", sorted(f for f in os.listdir("scripts") if f.startswith("09")))

## Cell 3 - install dependencies

In [ ]:
!pip -q install google-cloud-bigquery db-dtypes pandas pyarrow \
         xgboost shap scikit-learn matplotlib tabulate
print("installed")

## Cell 4 - get the data

Two routes. **Route A (faster, use if you have the file):** it is already on
your Drive at `hf-risk/mimic_hf_with_gcs.csv` (6.1 MB).

**Route B:** pull it fresh from BigQuery using script 08 (~40 s). Needs Colab
auth plus a Google account with approved PhysioNet MIMIC-IV access.

In [ ]:
import os, subprocess

ROUTE = "A"  # change to "B" to pull from BigQuery instead
os.makedirs("data", exist_ok=True)

if ROUTE == "A":
    from google.colab import drive
    drive.mount("/content/drive")
    src = "/content/drive/MyDrive/hf-risk/mimic_hf_with_gcs.csv"
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"{src} not found. Put mimic_hf_with_gcs.csv there, or set ROUTE='B'.")
    !cp "{src}" data/mimic_hf_with_gcs.csv
else:
    from google.colab import auth
    auth.authenticate_user()
    subprocess.run(["python", "scripts/08_mimic_gcs_extraction.py"], check=True)

import pandas as pd
df = pd.read_csv("data/mimic_hf_with_gcs.csv")
print("shape:", df.shape)
print("patients:", df.subject_id.nunique())
print("GCS coverage: %.1f%%" % (100 * df.gcs_total.notna().mean()))

## Cell 5 - smoke test (a few minutes)

Proves the whole code path works before you spend 15-40 minutes. Uses a
patient-level 50% subsample, no CV, tiny bootstrap, few trees.

In [ ]:
!python scripts/09_mimic_native_gcs_retrain.py --quick --no-cv \
        --bootstrap 50 --n-estimators 100

print("\n--- smoke test results ---")
print(pd.read_csv("results/tables/mimic_gcs_retrain_results.csv").to_string(index=False))

## Cell 6 - the real run

Expect roughly 15-40 minutes. Same script, full cohort, 5-fold GroupKFold CV,
1,000-iteration paired bootstrap, 400 trees, plus both TreeSHAP passes.

If you want the headline numbers faster and don't need the CV column:
uncomment the fast variant at the bottom.

In [ ]:
!python scripts/09_mimic_native_gcs_retrain.py

# Fast variant if you are short on time or credit:
# !python scripts/09_mimic_native_gcs_retrain.py --no-cv --bootstrap 200 \
#         --n-estimators 200

## Cell 7 - read the results

In [ ]:
res = pd.read_csv("results/tables/mimic_gcs_retrain_results.csv")
print(res.to_string(index=False))

print("\n=== RUN LOG ===")
print(open("results/mimic_gcs_run_log.md", encoding="utf-8").read())

## Cell 8 - SHAP importance, both cohorts

In [ ]:
for tag in ["full_cohort", "icu_only"]:
    f = f"results/tables/shap_importance_{tag}.csv"
    imp = pd.read_csv(f)
    print(f"\n=== {tag} - top 10 ===")
    print(imp.head(10).to_string(index=False))

from IPython.display import Image, display
import os
for tag in ["full_cohort", "icu_only"]:
    p = f"results/figures/shap_bar_{tag}.png"
    if os.path.exists(p):
        print(f"\n{tag}")
        display(Image(p))

## Cell 9 - save everything to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
base = "/content/drive/MyDrive/hf-risk"
!mkdir -p "{base}/results/tables" "{base}/results/figures" "{base}/outputs/mimic_gcs"
!cp results/mimic_gcs_run_log.md "{base}/results/"
!cp results/tables/*.csv "{base}/results/tables/"
!cp results/figures/shap_* "{base}/results/figures/"
!cp outputs/mimic_gcs/*.csv "{base}/outputs/mimic_gcs/"
print("saved to", base)

## Interpreting the output honestly

Four things to keep straight when you write this up:

1. **`expanded_baseline` is not the historical 0.801 model.** Only 10 of the
   legacy 24 columns exist in the GCS extract (see
   `results/tables/legacy_24_features.json`), so the baseline is a superset
   model. Do not label it 'the 0.801 model'.

2. **A CI that crosses zero is a null result.** If `delta_95ci_low` is below
   zero and `delta_95ci_high` is above it, write "no detectable improvement
   from adding GCS at this sample size" - that is a publishable, honest finding.

3. **`baseline_plus_icu_flag` is the control.** If it matches
   `expanded_plus_gcs`, the gain is about *being an ICU patient*, not
   neurological status. Report it either way - it is more honest, not a failure.

4. **`full` and `icu_only` are different populations.** The full cohort has GCS
   for 28% of admissions; the ICU-only cohort has it for 100%. Do not compare
   those AUROCs to each other. Decide deliberately which one leads the
   narrative - the full-cohort number is AUROC-comparable, the ICU-only number
   is SHAP-comparable. They answer different questions.